In [ ]:
import re
import html
import spacy # type: ignore
import numpy as np # type: ignore
import requests # type: ignore
import time
import os
import json
from collections import Counter
from mitreattack.stix20 import MitreAttackData # type: ignore
from dotenv import load_dotenv # type: ignore
from newspaper import Article # type: ignore


In [ ]:
def build_mitre_dict(mitre_json_path, nlp, top_n_keywords=10):

    # === Precompiled regex and cached filler words ===
    html_tag_re = re.compile(r'<.*?>')
    md_link_re = re.compile(r'\[([^\]]+)\]\([^)]+\)')
    inline_code_re = re.compile(r'`([^`]+)`')
    citation_re = re.compile(r'\(Citation:.*?\)')
    filler_words = re.compile(r'\b(adversaries|may|possibly|can|could|might)\b', re.IGNORECASE)

    def preprocess_description(desc):
        if not desc:
            return ""
        desc = html_tag_re.sub('', desc)
        desc = html.unescape(desc)
        desc = md_link_re.sub(r'\1', desc)
        desc = inline_code_re.sub(r'\1', desc)
        desc = citation_re.sub('', desc)
        desc = filler_words.sub('', desc)
        return re.sub(r'\s+', ' ', desc).strip()

    def extract_keywords(doc, top_n=top_n_keywords):
        # Minor optimization: use a set of stopwords for fast membership testing
        return [token.text.lower() for token in doc
                if token.is_alpha and not token.is_stop and len(token.text) > 2][:top_n]

    def get_vector(doc):
        ragged = doc._.trf_data.last_hidden_layer_state
        tensor = ragged.data
        return tensor.mean(0) if tensor.shape[0] > 0 else np.zeros((tensor.shape[1],))

    # === Load once ===
    with open(mitre_json_path, "r") as f:
        stix_data = json.load(f)
    stix_objects = stix_data["objects"]

    # === Actor Relationships ===
    relationships = [obj for obj in stix_objects if obj.get("type") == "relationship"]
    actors = {obj["id"]: obj["name"] for obj in stix_objects if obj.get("type") == "intrusion-set"}

    technique_to_actors = {}
    for rel in relationships:
        if rel.get("relationship_type") == "uses" and \
            rel["source_ref"].startswith("intrusion-set") and \
            rel["target_ref"].startswith("attack-pattern"):

            actor_name = actors.get(rel["source_ref"])
            if actor_name:
                technique_to_actors.setdefault(rel["target_ref"], []).append(actor_name)

    # === Load Techniques via MitreAttackData ===
    attack_data = MitreAttackData(mitre_json_path)
    techniques = attack_data.get_techniques()

    mitre_dict = {}
    for tech in techniques:
        external_id = next(
            (ref.get("external_id") for ref in tech.get("external_references", [])
                if ref.get("external_id", "").startswith("T")),
            None
        )
        if not external_id:
            continue

        description = preprocess_description(tech.get("description", ""))
        if not description:
            continue

        doc = nlp(description)
        keywords = extract_keywords(doc)

        mitre_dict[external_id] = {
            "name": tech.get("name", "Unnamed Technique"),
            "tactics": list({phase["phase_name"] for phase in tech.get("kill_chain_phases", [])}),
            "description": description,
            "keywords": keywords,
            "spacy_doc": doc,
            "vector": get_vector(doc),
            "actors": sorted(set(technique_to_actors.get(tech["id"], [])))
        }

    return mitre_dict


In [17]:
def process_text_to_mitre_tag(text, mitre_dict, nlp, threshold=0.9, keyword_boost=0.1):
    def get_vector(doc):
        ragged = doc._.trf_data.last_hidden_layer_state
        tensor = ragged.data
        if tensor.shape[0] == 0:
            # Return a zero vector of the correct dim (768 for `en_core_web_trf`)
            return np.zeros((ragged.data.shape[1],))
        return tensor.mean(0)


    def infer_tactic_hints(text):
        hints = []
        if "network" in text or "traffic" in text:
            hints.append("command-and-control")
        if "initially access" in text or "entry point" in text:
            hints.append("initial-access")
        return hints

    doc = nlp(text)
    vec = get_vector(doc)
    text_lower = text.lower()
    
    matches = []
    for external_id, data in mitre_dict.items():
        if vec.shape[0] == 0 or data["vector"].shape[0] == 0:
            continue  # skip similarity comparison if any vector is invalid

        sim = float(np.dot(vec, data["vector"]) /
                    (np.linalg.norm(vec) * np.linalg.norm(data["vector"])))
        if np.isnan(sim):
            continue

        if any(k.lower() in text_lower for k in data["keywords"]):
            sim += keyword_boost

        keyword_match = sum(1 for k in data["keywords"] if k in text_lower)
        tactic_match = sum(1 for t in data["tactics"] if t in infer_tactic_hints(text_lower))
        final_score = round(sim + (keyword_boost * keyword_match) + (0.05 * tactic_match), 2)

        if sim >= threshold:
            matches.append({
                "external_id": external_id,
                "tactics": data["tactics"],
                "name": data["name"],
                "similarity": round(sim, 3),
                "final_score": final_score
            })

    matches.sort(key=lambda x: x["final_score"], reverse=True)
    return matches[:5]  # Top 5 matches (or fewer)


In [ ]:
def extract_text_from_url(url):
    article = Article(url)
    article.download()
    article.parse()
    return article.title, article.text

def extract_keywords(doc, top_n=10):
        tokens = [token.text.lower() for token in doc
                    if token.is_alpha and not token.is_stop and len(token.text) > 2]
        return [word for word, _ in Counter(tokens).most_common(top_n)]
    
def tag_articles(url_list, mitre_dict, nlp):
    results = []
    for url in url_list:
        try:
            text = extract_text_from_url(url)
            match = process_text_to_mitre_tag(text, mitre_dict, nlp)
            results.append({
                "url": url,
                "external_id": match["external_id"],
                "tactics": match["tactics"]
            })
        except Exception as e:
            results.append({"url": url, "error": str(e)})
    return results


In [ ]:
# Load API credentials securely from .env file
load_dotenv(dotenv_path="GoogleApi.env")
API_KEY = os.getenv("GOOGLE_API_KEY")
CX = os.getenv("GOOGLE_CX")

if not API_KEY or not CX:
    raise ValueError("Missing Google API Key or Custom Search Engine ID!")

# Search queries for threat intel
THREAT_QUERIES = [
    "medusa malware:2025-01-01"
]

# Global NLP + MITRE dict setup
if "mitre_dict" not in globals():
    nlp = spacy.load("en_core_web_trf")
    mitre_dict = build_mitre_dict(
        '/Users/jaytlinaskew/Documents/MainDocuments/Alyn/Models/Documentation/Natural Language Processing Packages/enterprise-attack.json',
        nlp
)

# === Helper Functions ===

def extract_text_from_url(url):
    article = Article(url)
    article.download()
    article.parse()
    return article.title, article.text

def extract_keywords(doc, top_n=50):
    tokens = [token.text.lower() for token in doc
                if token.is_alpha and not token.is_stop and len(token.text) > 2]
    return [word for word, _ in Counter(tokens).most_common(top_n)]

def tag_mitre_from_url(url):
    try:
        title, text = extract_text_from_url(url)
        doc = nlp(text)
        article_keywords = set(extract_keywords(doc, top_n=50))
        matches = process_text_to_mitre_tag(text, mitre_dict, nlp)

        for match in matches:
            mitre_keywords = set(mitre_dict[match['external_id']]['keywords'])
            match["matching_keywords"] = sorted(article_keywords & mitre_keywords)

        return {
            "url": url,
            "title": title,
            "matches": matches
        }

    except Exception as e:
        return {
            "url": url,
            "error": str(e)
        }

# === Main Threat Intelligence Fetch + Tag ===

def fetch_threat_intelligence():
    for query in THREAT_QUERIES:
        print(f"\n🔍 Searching: {query}")
        formatted_query = query.replace(" ", "+")
        url = f"https://www.googleapis.com/customsearch/v1?q={formatted_query}&key={API_KEY}&cx={CX}&num=2" # num controls how many resources we pull per query

        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()

            if "items" in data:
                for item in data["items"]:
                    article_url = item.get("link", "")
                    print(f"\n Processing article: {article_url}")

                    tagged = tag_mitre_from_url(article_url)

                    if "error" in tagged:
                        print(f" Error: {tagged['error']}")
                    elif not tagged["matches"]:
                        print("No technique matches found.")
                    else:
                        print(f"Title: {tagged['title']}")
                        for i, match in enumerate(tagged["matches"], 1):
                            print(f"--- Match {i} ---")
                            print(f"Technique ID     : {match['external_id']}")
                            print(f"Name             : {match['name']}")
                            print(f"Tactics          : {', '.join(match['tactics'])}")
                            print(f"Score            : {match['final_score']}")
                            print(f"Keywords         : {', '.join(mitre_dict[match['external_id']]['keywords'])}")
                            print(f"Matching Keywords: {', '.join(match['matching_keywords']) or 'None'}\n")
                            print(f"Threat Actors    : {', '.join(mitre_dict[match['external_id']].get('actors', [])) or 'None'}\n")
            else:
                print(f"No results for: {query}")

        except requests.exceptions.RequestException as e:
            print(f"API Request Failed: {e}")

        time.sleep(2)


In [ ]:
if __name__ == "__main__":
    print(" Starting open source threat intel tagging...\n")
    fetch_threat_intelligence()


 Starting automated threat intel tagging...


🔍 Searching: medusa malware

 Processing article: https://www.tripwire.com/state-of-security/medusa-ransomware-what-you-need-know
Title: Medusa Ransomware: What You Need To Know
--- Match 1 ---
Technique ID     : T1486
Name             : Data Encrypted for Impact
Tactics          : impact
Score            : 1.96
Keywords         : data, files, network, system, decryption, key, like, encrypt, target, systems
Matching Keywords: data, files, like, network, systems

Threat Actors    : APT38, APT41, Akira, FIN7, FIN8, INC Ransom, Indrik Spider, Magic Hound, Moonstone Sleet, Sandworm Team, Scattered Spider, TA505

--- Match 2 ---
Technique ID     : T1564.005
Name             : Hidden File System
Tactics          : defense-evasion
Score            : 1.96
Keywords         : file, system, systems, standard, disk, use, hidden, malicious, security, tools
Matching Keywords: malicious, systems, use

Threat Actors    : Equation, Strider

--- Match 3 ---
